In [13]:
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

epsilon = 0.0
gamma_0 = 1.0
U_0 = 0.0

num_sites = 6
num_spin_orbitals = 2*num_sites

terms = {}
for i in range(num_spin_orbitals):
    terms[f"+_{i} -_{i}"] = epsilon
hopping_pairs = [(i, (i+1) % 6) for i in range(num_sites)]
for i, j in hopping_pairs:
    i_up, i_down, j_up, j_down = i, num_sites+i, j, num_sites+j
    terms[f"+_{i_up} -_{j_up}"] = terms[f"+_{j_up} -_{i_up}"] = terms[f"+_{i_down} -_{j_down}"] = terms[f"+_{j_down} -_{i_down}"] = -gamma_0
for i in range(num_sites):
    up, down = i, num_sites+i
    terms[f"+_{up} -_{up} +_{down} -_{down}"] = U_0

fermionic_hamiltonian = FermionicOp(terms, num_spin_orbitals)
jw = JordanWignerMapper()
qubit_hamiltonian = jw.map(fermionic_hamiltonian)

In [16]:
from qiskit.synthesis import SuzukiTrotter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit_nature.second_q.circuit.library import HartreeFock

t = 0.2
N_trot = 15
st = SuzukiTrotter(reps=N_trot)
evolution = PauliEvolutionGate(qubit_hamiltonian, time=t)
evolution_circuit = st.synthesize(evolution)
hf_circuit = HartreeFock(num_spatial_orbitals=num_sites, num_particles=(5, 3), qubit_mapper=jw)

In [17]:
from qiskit_algorithms import IterativePhaseEstimation
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Sampler
from qiskit import transpile
import numpy as np

backend, sampler = AerSimulator(), Sampler()
evolution_circuit, hf_circuit = transpile(evolution_circuit, backend=backend), transpile(hf_circuit, backend=backend)
iqpe = IterativePhaseEstimation(num_iterations=5, sampler=sampler)
result = iqpe.estimate(unitary=evolution_circuit, state_preparation=hf_circuit)

energy = -2*np.pi * result.phase / t
print(energy)

-30.434178831651117
